# Synthea CSV integration: complete walkthrough

**Prerequisite:** Java 17+ must be installed (on `PATH`, via `JAVA_HOME`, or passed as `java_path=`). PyHealth does not install it. You do not need to download Synthea itself: the pinned JAR is downloaded and checksum-verified on the first `generate()` call. See the *Setup* section of the `pyhealth.models.Synthea` API docs for Java install commands and offline use.

This notebook walks through the CSV integration:

- discovering supported `synthea.properties` before generating anything;
- setting up one `Synthea` generator and previewing its command;
- generating several populations from that one instance, with reuse and `overwrite`;
- every `generate()` argument and the Synthea CLI flag it maps to;
- loading generated CSV through `SyntheaCSVDataset`; and
- how Synthea data differs from other PyHealth datasets.

## Prerequisites

Java 17 or newer and, on first use, internet access to download the pinned Synthea JAR (about 200 MB) unless `jar_path` is supplied. Each live generation below takes roughly 15 seconds for a handful of patients.

In [1]:
import inspect
import shlex
from pathlib import Path

from pyhealth.datasets import SyntheaCSVDataset
from pyhealth.models import Synthea

In [2]:
artifact_root = Path(".artifacts/synthea_csv").resolve()
artifact_root.mkdir(parents=True, exist_ok=True)
artifact_root

PosixPath('/home/soham/PyHealth/examples/.artifacts/synthea_csv')

## Public generator API

| Member | Purpose |
|---|---|
| `Synthea(output_dir, ...)` | Set up the generator: output location, Java and JAR discovery, timeout. Generates nothing. |
| `generate(...)` | Generate a population from per-call settings and return its CSV directory; reuses existing output for identical settings unless `overwrite=True` |
| `build_argv(java, jar, ...)` | Build the Java/Synthea command for given settings without running it |
| `get_available_config()` | Return properties supported by the selected Synthea JAR |

In [3]:
{
    "Synthea": str(inspect.signature(Synthea)),
    "generate": str(inspect.signature(Synthea.generate)),
    "build_argv": str(inspect.signature(Synthea.build_argv)),
    "get_available_config": str(inspect.signature(Synthea.get_available_config)),
}

{'Synthea': "(output_dir: 'str | Path', java_path: 'str | Path | None' = None, jar_path: 'str | Path | None' = None, auto_download: 'bool' = True, timeout: 'float | None' = None) -> 'None'",
 'generate': "(self, population: 'int | None' = None, seed: 'int | None' = None, state: 'str | None' = None, city: 'str | None' = None, clinician_seed: 'int | None' = None, single_person_seed: 'int | None' = None, reference_date: 'str | None' = None, end_date: 'str | None' = None, gender: 'str | None' = None, age_range: 'str | None' = None, overflow_population: 'bool | None' = None, local_config_path: 'str | Path | None' = None, local_modules_dir: 'str | Path | None' = None, initial_population_snapshot_path: 'str | Path | None' = None, updated_population_snapshot_path: 'str | Path | None' = None, update_time_period: 'int | None' = None, fixed_record_path: 'str | Path | None' = None, keep_matching_patients_path: 'str | Path | None' = None, synthea_config: 'Mapping[str, str | int | float | bool] | No

## 1. Discover supported Synthea configuration

These class methods work before a generator exists. `pattern` uses shell-style glob matching. Discovery reads `synthea.properties` from the actual pinned or explicitly supplied JAR, so it reflects the Synthea version that will run.

In [4]:
generation_properties = Synthea.get_available_config(
    pattern="generate.*"
)
list(sorted(generation_properties.items()))[:10]

[('generate.append_numbers_to_person_names', 'true'),
 ('generate.birthweights.default_file', 'birthweights.csv'),
 ('generate.birthweights.logging', 'false'),
 ('generate.costs.default_device_cost', '0.00'),
 ('generate.costs.default_encounter_cost', '125.00'),
 ('generate.costs.default_immunization_cost', '136.00'),
 ('generate.costs.default_lab_cost', '100.00'),
 ('generate.costs.default_medication_cost', '255.00'),
 ('generate.costs.default_procedure_cost', '500.00'),
 ('generate.costs.default_supply_cost', '0.00')]

The examples below use several ordinary Synthea properties:

- `generate.thread_pool_size=1` makes generation serial, so output is deterministic for a fixed seed;
- `generate.only_alive_patients=true` keeps only patients alive at the simulation end;
- `exporter.years_of_history=5` limits exported history;
- `exporter.csv.folder_per_run=false` keeps CSV files directly under the CSV directory; and
- `exporter.csv.append_mode=false` replaces rather than appends to prior CSV files.

CSV exporter selection and the output directory are managed by the wrapper and are rejected if placed in `synthea_config`.

## 2. Set up the generator

The constructor holds only setup. Population settings are passed to `generate()`, so one instance can produce any number of populations.

In [5]:
synthea = Synthea(output_dir=artifact_root / "generated", timeout=900)
synthea

Synthea(output_dir='/home/soham/PyHealth/examples/.artifacts/synthea_csv/generated')

`build_argv()` shows the exact command `generate()` would run for some settings, without needing Java. Each distinct combination of settings is written to its own fingerprinted directory under `output_dir`, visible as `--exporter.baseDirectory`.

In [6]:
common_config = {
    "generate.thread_pool_size": 1,
    "generate.only_alive_patients": True,
    "exporter.years_of_history": 5,
    "exporter.csv.folder_per_run": False,
    "exporter.csv.append_mode": False,
}
settings = dict(
    population=10,
    seed=42,
    clinician_seed=43,
    reference_date="20240101",
    gender="F",
    age_range="30-40",
    overflow_population=False,
    state="Massachusetts",
    city="Boston",
    synthea_config=common_config,
)

preview_argv = synthea.build_argv(Path("java"), Path("synthea.jar"), **settings)
print(shlex.join(preview_argv))

java -jar synthea.jar --exporter.baseDirectory=/home/soham/PyHealth/examples/.artifacts/synthea_csv/generated/3f9629628a6636e7bcd7cdac7b8a158b56c8732ceaa87faf747775c20b85f8e4 --exporter.bfd.export=false --exporter.ccda.export=false --exporter.cdw.export=false --exporter.clinical_note.export=false --exporter.cpcds.export=false --exporter.csv.append_mode=false --exporter.csv.export=true --exporter.csv.folder_per_run=false --exporter.fhir.export=false --exporter.fhir_dstu2.export=false --exporter.fhir_stu3.export=false --exporter.json.export=false --exporter.symptoms.csv.export=false --exporter.symptoms.text.export=false --exporter.text.export=false --exporter.years_of_history=5 --generate.only_alive_patients=true --generate.thread_pool_size=1 -s 42 -cs 43 -p 10 -r 20240101 -g F -a 30-40 -o false Massachusetts Boston


## 3. Generate several populations from one instance

Each call runs Synthea once and returns the CSV directory. Different settings go to different directories, so populations never overwrite each other.

In [7]:
csv_path = synthea.generate(**settings)
smaller_path = synthea.generate(population=3, seed=7, synthea_config=common_config)

assert csv_path != smaller_path
assert (csv_path / "patients.csv").is_file()
assert (smaller_path / "patients.csv").is_file()
sorted(path.name for path in csv_path.glob("*.csv"))

Running Synthea: /home/soham/.local/jvm/jdk-17.0.20.1+1/bin/java -jar /home/soham/.cache/pyhealth/synthea/synthea-with-dependencies-v4.0.0.jar --exporter.baseDirectory=/home/soham/PyHealth/examples/.artifacts/synthea_csv/generated/3f9629628a6636e7bcd7cdac7b8a158b56c8732ceaa87faf747775c20b85f8e4 --exporter.bfd.export=false --exporter.ccda.export=false --exporter.cdw.export=false --exporter.clinical_note.export=false --exporter.cpcds.export=false --exporter.csv.append_mode=false --exporter.csv.export=true --exporter.csv.folder_per_run=false --exporter.fhir.export=false --exporter.fhir_dstu2.export=false --exporter.fhir_stu3.export=false --exporter.json.export=false --exporter.symptoms.csv.export=false --exporter.symptoms.text.export=false --exporter.text.export=false --exporter.years_of_history=5 --generate.only_alive_patients=true --generate.thread_pool_size=1 -s 42 -cs 43 -p 10 -r 20240101 -g F -a 30-40 -o false Massachusetts Boston


org.mitre.synthea.X12.ExporterAdaptor
org.mitre.synthea.X12.ExporterAdaptor


SLF4J: No SLF4J providers were found.
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J: See http://www.slf4j.org/codes.html#noProviders for further details.


Scanned 92 modules and 449 submodules.
Loading submodule modules/allergies/allergy_panel.json
Loading submodule modules/allergies/drug_allergy_incidence.json


Loading submodule modules/allergies/environmental_allergy_incidence.json
Loading submodule modules/allergies/food_allergy_incidence.json
Loading submodule modules/allergies/immunotherapy.json
Loading submodule modules/allergies/outgrow_env_allergies.json
Loading submodule modules/allergies/outgrow_food_allergies.json
Loading submodule modules/allergies/severe_allergic_reaction.json
Loading submodule modules/anemia/anemia_sub.json


...

Loading module modules/veteran_self_harm.json
Loading module modules/veteran_substance_abuse_conditions.json
Loading module modules/veteran_substance_abuse_treatment.json
Loading module modules/vhd_aortic.json
Loading Lookup Table: vhd_as.csv
Loading Lookup Table: vhd_ar.csv
Loading module modules/vhd_mitral.json
Loading Lookup Table: vhd_mr.csv
Loading Lookup Table: vhd_ms.csv
Loading module modules/vhd_pulmonic.json
Loading Lookup Table: vhd_ps.csv
Loading Lookup Table: vhd_pr.csv
Loading module modules/vhd_tricuspid.json
Loading Lookup Table: vhd_tr.csv
Loading Lookup Table: vhd_ts.csv
Loading module modules/wellness_encounters.json
Running with options:
Population: 3
Seed: 7
Provider Seed:1791484082286
Reference Time: 1791484082286
Location: Massachusetts
Min Age: 0
Max Age: 140


1 -- Darron84 Hegmann834 (36 y/o M) Salem, Massachusetts  (50652)


2 -- Samara159 Runolfsdottir785 (14 y/o F) Chelsea, Massachusetts  (20570)
3 -- Gwyn381 Runolfsson901 (22 y/o F) North Attleborough, Massachusetts  (31764)


Records: total=3, alive=3, dead=0
RNG=3
Clinician RNG=5640

You've just generated 3 patients! We're happy to know Synthea is helping you with your project.
Like what we're doing? Synthea is an open-source project, and your support helps us keep building.
Please consider ]8;;https://github.com/sponsors/synthetichealthmaking a donation]8;; (https://github.com/sponsors/synthetichealth).



['allergies.csv',
 'careplans.csv',
 'claims.csv',
 'claims_transactions.csv',
 'conditions.csv',
 'devices.csv',
 'encounters.csv',
 'imaging_studies.csv',
 'immunizations.csv',
 'medications.csv',
 'observations.csv',
 'organizations.csv',
 'patients.csv',
 'payer_transitions.csv',
 'payers.csv',
 'procedures.csv',
 'providers.csv',
 'supplies.csv']

Calling `generate()` again with identical settings returns the existing directory immediately instead of rerunning Synthea. `overwrite=True` forces a fresh run into the same directory. If `seed` is omitted, a random seed is chosen and logged, so each seedless call is a new population.

In [8]:
reused_path = synthea.generate(**settings)
assert reused_path == csv_path

overwritten_path = synthea.generate(**settings, overwrite=True)
assert overwritten_path == csv_path

Reusing existing Synthea output at /home/soham/PyHealth/examples/.artifacts/synthea_csv/generated/3f9629628a6636e7bcd7cdac7b8a158b56c8732ceaa87faf747775c20b85f8e4/csv


Running Synthea: /home/soham/.local/jvm/jdk-17.0.20.1+1/bin/java -jar /home/soham/.cache/pyhealth/synthea/synthea-with-dependencies-v4.0.0.jar --exporter.baseDirectory=/home/soham/PyHealth/examples/.artifacts/synthea_csv/generated/3f9629628a6636e7bcd7cdac7b8a158b56c8732ceaa87faf747775c20b85f8e4 --exporter.bfd.export=false --exporter.ccda.export=false --exporter.cdw.export=false --exporter.clinical_note.export=false --exporter.cpcds.export=false --exporter.csv.append_mode=false --exporter.csv.export=true --exporter.csv.folder_per_run=false --exporter.fhir.export=false --exporter.fhir_dstu2.export=false --exporter.fhir_stu3.export=false --exporter.json.export=false --exporter.symptoms.csv.export=false --exporter.symptoms.text.export=false --exporter.text.export=false --exporter.years_of_history=5 --generate.only_alive_patients=true --generate.thread_pool_size=1 -s 42 -cs 43 -p 10 -r 20240101 -g F -a 30-40 -o false Massachusetts Boston


org.mitre.synthea.X12.ExporterAdaptor
org.mitre.synthea.X12.ExporterAdaptor


SLF4J: No SLF4J providers were found.
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J: See http://www.slf4j.org/codes.html#noProviders for further details.


Scanned 92 modules and 449 submodules.
Loading submodule modules/allergies/allergy_panel.json
Loading submodule modules/allergies/drug_allergy_incidence.json


...

4 -- Esperanza675 María Luisa888 Valverde277 (32 y/o F) Boston, Massachusetts  (45515)
5 -- Jona712 Bonnie873 Rosenbaum794 (35 y/o F) Boston, Massachusetts  (51640)


6 -- Tennille443 Denna536 Nitzsche158 (36 y/o F) Boston, Massachusetts  (49197)
7 -- Ocie984 Bernie827 Homenick806 (40 y/o F) Boston, Massachusetts  (56300)


8 -- Lindsay928 Deja232 Pollich983 (38 y/o F) Boston, Massachusetts  (52138)
9 -- Shondra529 Kyra237 Spinka232 (34 y/o F) Boston, Massachusetts  (46387)


10 -- Chrissy459 Jolynn62 Stanton715 (40 y/o F) Boston, Massachusetts  (58930)


Records: total=10, alive=10, dead=0
RNG=10
Clinician RNG=5638

You've just generated 10 patients! We're happy to know Synthea is helping you with your project.
Like what we're doing? Synthea is an open-source project, and your support helps us keep building.
Please consider ]8;;https://github.com/sponsors/synthetichealthmaking a donation]8;; (https://github.com/sponsors/synthetichealth).



## 4. Every `generate()` argument

This is a **dry-run API example** using `build_argv()`. Snapshot, fixed-record, keep-matching, and custom-module inputs require domain-specific files, so these settings are only used to show argument coverage and inspect the command. Do not pass them to `generate()` until the placeholder files are replaced with valid Synthea inputs.

Passing `None` means that option is omitted and Synthea uses its own default. When both `local_config_path` and `synthea_config` define a property, the mapping is emitted later as a command-line override and therefore wins.

In [9]:
demo_root = artifact_root / "argument_demo"
demo_root.mkdir(parents=True, exist_ok=True)
local_config_path = demo_root / "local.properties"
local_config_path.write_text(
    "generate.only_alive_patients = true\n"
    "exporter.years_of_history = 5\n",
    encoding="utf-8",
)

all_generate_args = {
    "population": 25,
    "seed": 1001,
    "state": "Massachusetts",
    "city": "Boston",
    "clinician_seed": 1002,
    "single_person_seed": 1003,
    "reference_date": "20240101",
    "end_date": "20241231",
    "gender": "F",
    "age_range": "18-65",
    "overflow_population": False,
    "local_config_path": local_config_path,
    "local_modules_dir": demo_root / "modules",
    "initial_population_snapshot_path": demo_root / "initial.snapshot",
    "updated_population_snapshot_path": demo_root / "updated.snapshot",
    "update_time_period": 30,
    "fixed_record_path": demo_root / "fixed_records.json",
    "keep_matching_patients_path": demo_root / "keep_module.json",
    "synthea_config": {
        "generate.thread_pool_size": 1,
        "generate.only_alive_patients": False,
        "exporter.csv.folder_per_run": False,
        "exporter.csv.append_mode": False,
    },
}

declared_args = set(inspect.signature(Synthea.generate).parameters) - {"self", "overwrite"}
assert set(all_generate_args) == declared_args
all_args_argv = synthea.build_argv(Path("java"), Path("synthea.jar"), **all_generate_args)
assert all_args_argv.index("-c") < all_args_argv.index("--generate.only_alive_patients=false")
print(shlex.join(all_args_argv))

java -jar synthea.jar -c /home/soham/PyHealth/examples/.artifacts/synthea_csv/argument_demo/local.properties --exporter.baseDirectory=/home/soham/PyHealth/examples/.artifacts/synthea_csv/generated/1aa325cf3d707300d9d6e663cbaca600eaed444a06b7bf49988a6639194cfcf2 --exporter.bfd.export=false --exporter.ccda.export=false --exporter.cdw.export=false --exporter.clinical_note.export=false --exporter.cpcds.export=false --exporter.csv.append_mode=false --exporter.csv.export=true --exporter.csv.folder_per_run=false --exporter.fhir.export=false --exporter.fhir_dstu2.export=false --exporter.fhir_stu3.export=false --exporter.json.export=false --exporter.symptoms.csv.export=false --exporter.symptoms.text.export=false --exporter.text.export=false --generate.only_alive_patients=false --generate.thread_pool_size=1 -s 1001 -cs 1002 -ps 1003 -p 25 -r 20240101 -e 20241231 -g F -a 18-65 -o false -d /home/soham/PyHealth/examples/.artifacts/synthea_csv/argument_demo/modules -i /home/soham/PyHealth/examples/.

`generate()` argument to Synthea CLI mapping:

| `generate()` argument | Synthea CLI | Meaning |
|---|---|---|
| `seed` | `-s` | Population random seed; random and logged when omitted |
| `clinician_seed` | `-cs` | Clinician generation seed |
| `single_person_seed` | `-ps` | Seed used to reproduce a particular person |
| `population` | `-p` | Requested population size |
| `reference_date` | `-r` | Reference date in `YYYYMMDD` form |
| `end_date` | `-e` | Simulation end date in `YYYYMMDD` form |
| `gender` | `-g` | Restrict generation to `M` or `F` |
| `age_range` | `-a` | Restrict age using `min-max` |
| `overflow_population` | `-o` | Permit extra generated people to satisfy filters |
| `local_config_path` | `-c` | Additional `.properties` configuration file |
| `local_modules_dir` | `-d` | Directory containing custom Synthea modules |
| `initial_population_snapshot_path` | `-i` | Load a previously saved population snapshot |
| `updated_population_snapshot_path` | `-u` | Write the population after an update run |
| `update_time_period` | `-t` | Number of days advanced during a snapshot update |
| `fixed_record_path` | `-f` | Fixed-demographics input used to generate specified people |
| `keep_matching_patients_path` | `-k` | Module used to retain only matching patients |
| `state`, `city` | trailing positional arguments | Geographic generation location |
| `synthea_config` | `--property=value` overrides | Arbitrary supported `synthea.properties` values |
| `overwrite` | wrapper-managed | Rerun Synthea even if output for these settings exists |

Constructor arguments are all wrapper-managed setup:

| Constructor argument | Meaning |
|---|---|
| `output_dir` | Parent of the fingerprinted generation directories |
| `java_path` | Explicit Java executable, otherwise discovered |
| `jar_path` | Explicit Synthea JAR, otherwise the pinned release is used |
| `auto_download` | Allow downloading the pinned JAR when absent |
| `timeout` | Subprocess timeout in seconds for each `generate()` call |

## 5. Load generated CSV through PyHealth

`SyntheaCSVDataset` only reads files: it takes the CSV directory returned by `generate()` plus an optional table selection. Omitting `tables` uses all supported defaults. `dataset_name`, `config_path`, and normal `BaseDataset` keyword arguments such as `cache_dir` and `dev` can also be supplied.

In [10]:
dataset = SyntheaCSVDataset(
    root=csv_path,
    tables=["patients", "encounters", "conditions", "observations"],
    dataset_name="synthea_csv_demo",
    config_path=None,
    cache_dir=artifact_root / "cache",
    dev=False,
)

events = dataset.load_data()
dataset.stats()
# events.head(5)
events

Initializing synthea_csv_demo dataset from /home/soham/PyHealth/examples/.artifacts/synthea_csv/generated/3f9629628a6636e7bcd7cdac7b8a158b56c8732ceaa87faf747775c20b85f8e4/csv (dev mode: False)


Using provided cache_dir: /home/soham/PyHealth/examples/.artifacts/synthea_csv/cache/fa318dd5-8d48-5265-84fa-1d581ce4cd9a


Scanning table: patients from /home/soham/PyHealth/examples/.artifacts/synthea_csv/generated/3f9629628a6636e7bcd7cdac7b8a158b56c8732ceaa87faf747775c20b85f8e4/csv/patients.csv


Scanning table: encounters from /home/soham/PyHealth/examples/.artifacts/synthea_csv/generated/3f9629628a6636e7bcd7cdac7b8a158b56c8732ceaa87faf747775c20b85f8e4/csv/encounters.csv


Scanning table: conditions from /home/soham/PyHealth/examples/.artifacts/synthea_csv/generated/3f9629628a6636e7bcd7cdac7b8a158b56c8732ceaa87faf747775c20b85f8e4/csv/conditions.csv


...

Scanning table: observations from /home/soham/PyHealth/examples/.artifacts/synthea_csv/generated/3f9629628a6636e7bcd7cdac7b8a158b56c8732ceaa87faf747775c20b85f8e4/csv/observations.csv


Caching event dataframe to /home/soham/PyHealth/examples/.artifacts/synthea_csv/cache/fa318dd5-8d48-5265-84fa-1d581ce4cd9a/global_event_df.parquet...


2026-10-08 18:28:31,163 - distributed.core - ERROR - Exception while handling op terminate
Traceback (most recent call last):
  File "/home/soham/PyHealth/.venv/lib/python3.13/site-packages/distributed/core.py", line 834, in _handle_comm
    result = await result
             ^^^^^^^^^^^^
  File "/home/soham/PyHealth/.venv/lib/python3.13/site-packages/distributed/scheduler.py", line 4367, in close
    self.stop_services()
    ~~~~~~~~~~~~~~~~~~^^
  File "/home/soham/PyHealth/.venv/lib/python3.13/site-packages/distributed/node.py", line 79, in stop_services
    application.stop()
    ~~~~~~~~~~~~~~~~^^
  File "/home/soham/PyHealth/.venv/lib/python3.13/site-packages/bokeh/server/tornado.py", line 761, in stop
    raise RuntimeError("Cannot synchronously wait on a running event loop; use 'await stop_async()'")
RuntimeError: Cannot synchronously wait on a running event loop; use 'await stop_async()'


Dataset: synthea_csv_demo
Dev mode: False
Number of patients: 10
Number of events: 2778


,patient_id,event_type,timestamp,patients/birthdate,patients/deathdate,patients/gender,patients/race,patients/ethnicity,patients/marital,patients/birthplace,patients/city,patients/state,patients/county,patients/zip,encounters/id,encounters/start,encounters/stop,encounters/encounterclass,encounters/code,encounters/description,encounters/base_encounter_cost,encounters/total_claim_cost,encounters/payer_coverage,encounters/reasoncode,encounters/reasondescription,conditions/start,conditions/stop,conditions/encounter,conditions/code,conditions/description,observations/date,observations/encounter,observations/code,observations/description,observations/value,observations/units,observations/type
npartitions=4,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
,string,object,datetime64[ms],string,string,string,string,string,string,string,string,string,string,string,string,string,string,string,string,string,string,string,string,string,string,string,string,string,string,string,string,string,string,string,string,string,string
,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...


In [11]:
patients_table = dataset.load_table("patients")
patient_ids = dataset.unique_patient_ids
patient = dataset.get_patient(patient_ids[0])
patient_events = patient.get_events(return_df=True)

print("Patients:", len(patient_ids))
print("Selected patient:", patient.patient_id)
print("Configured tables:", dataset.tables)
patient_events.head(10)

Scanning table: patients from /home/soham/PyHealth/examples/.artifacts/synthea_csv/generated/3f9629628a6636e7bcd7cdac7b8a158b56c8732ceaa87faf747775c20b85f8e4/csv/patients.csv


Found 10 unique patient IDs


Patients: 10
Selected patient: aee7bbe1-0c45-c028-1e62-1f4cdb30c273
Configured tables: ['patients', 'encounters', 'conditions', 'observations']


patient_id,event_type,timestamp,patients/birthdate,patients/deathdate,patients/gender,patients/race,patients/ethnicity,patients/marital,patients/birthplace,patients/city,patients/state,patients/county,patients/zip,encounters/id,encounters/start,encounters/stop,encounters/encounterclass,encounters/code,encounters/description,encounters/base_encounter_cost,encounters/total_claim_cost,encounters/payer_coverage,encounters/reasoncode,encounters/reasondescription,conditions/start,conditions/stop,conditions/encounter,conditions/code,conditions/description,observations/date,observations/encounter,observations/code,observations/description,observations/value,observations/units,observations/type
str,str,datetime[ms],str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""aee7bbe1-0c45-c028-1e62-1f4cdb…","""patients""",1985-06-04 00:00:00,"""1985-06-04""",null,"""F""","""white""","""hispanic""","""M""","""Gaudalajara Jalisco MX""","""Boston""","""Massachusetts""","""Suffolk County""","""02114""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""aee7bbe1-0c45-c028-1e62-1f4cdb…","""conditions""",1995-05-08 00:00:00,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""1995-05-08""",null,"""aee7bbe1-0c45-c028-4bd4-fdd4de…","""40055000""","""Chronic sinusitis (disorder)""",null,null,null,null,null,null,null
"""aee7bbe1-0c45-c028-1e62-1f4cdb…","""encounters""",1995-05-08 08:15:37,null,null,null,null,null,null,null,null,null,null,null,"""aee7bbe1-0c45-c028-4bd4-fdd4de…","""1995-05-08T08:15:37Z""","""1995-05-08T08:30:37Z""","""ambulatory""","""185345009""","""Encounter for symptom (procedu…","""85.55""","""92.93""","""55.76""","""75498004""","""Acute bacterial sinusitis (dis…",null,null,null,null,null,null,null,null,null,null,null,null
"""aee7bbe1-0c45-c028-1e62-1f4cdb…","""conditions""",2003-07-29 00:00:00,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""2003-07-29""",null,"""aee7bbe1-0c45-c028-fc17-89784f…","""266934004""","""Transport problem (finding)""",null,null,null,null,null,null,null
"""aee7bbe1-0c45-c028-1e62-1f4cdb…","""conditions""",2003-07-29 00:00:00,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""2003-07-29""",null,"""aee7bbe1-0c45-c028-fc17-89784f…","""224299000""","""Received higher education (fin…",null,null,null,null,null,null,null
"""aee7bbe1-0c45-c028-1e62-1f4cdb…","""encounters""",2003-07-29 06:15:37,null,null,null,null,null,null,null,null,null,null,null,"""aee7bbe1-0c45-c028-fc17-89784f…","""2003-07-29T06:15:37Z""","""2003-07-29T07:12:15Z""","""wellness""","""162673000""","""General examination of patient…","""136.80""","""919.90""","""0.00""",null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""aee7bbe1-0c45-c028-1e62-1f4cdb…","""conditions""",2004-08-03 00:00:00,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""2004-08-03""",null,"""aee7bbe1-0c45-c028-1f6f-b01430…","""713458007""","""Lack of access to transportati…",null,null,null,null,null,null,null
"""aee7bbe1-0c45-c028-1e62-1f4cdb…","""encounters""",2004-08-03 06:15:37,null,null,null,null,null,null,null,null,null,null,null,"""aee7bbe1-0c45-c028-1f6f-b01430…","""2004-08-03T06:15:37Z""","""2004-08-03T07:02:44Z""","""wellness""","""162673000""","""General examination of patient…","""136.80""","""919.90""","""0.00""",null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""aee7bbe1-0c45-c028-1e62-1f4cdb…","""conditions""",2008-07-29 00:00:00,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""2008-07-29""",null,"""aee7bbe1-0c45-c028-f436-cec318…","""161744009""","""Past pregnancy h

## 6. How Synthea differs from other PyHealth datasets

The event table above has the same shape as every PyHealth dataset: one row per event with `patient_id`, `event_type` (the source table), `timestamp`, and one `<table>/<column>` column per attribute, null for other tables. What differs is the raw input and the codes.

**Raw output.** Synthea wrote the 18 CSV files listed in section 3, each wide and keyed by UUIDs, including billing (`claims`, `claims_transactions`) and institution tables (`organizations`, `providers`, `payers`). `SyntheaCSVDataset` loads the patient-keyed clinical tables by default; `claims` and `claims_transactions` load only when listed in `tables`; institution tables have no patient column and are never loaded.

**Codes.** MIMIC uses ICD-9/10 and NDC. Synthea uses:

| Table | Code system | Example `code` |
|---|---|---|
| `conditions`, `procedures`, `encounters`, `careplans`, `devices`, `supplies` | SNOMED-CT | `714628002` (Prediabetes) |
| `medications` | RxNorm | `849574` (Naproxen sodium 220 MG Oral Tablet) |
| `observations` | LOINC | `4548-4` (Hemoglobin A1c) |
| `immunizations` | CVX | `140` (Influenza, split virus, trivalent, PF) |

So MIMIC tasks such as `MortalityPredictionMIMIC3` and ICD-based `pyhealth.medcode` mappings do not apply; write a Synthea-specific task instead. Every code has a human-readable `description`.

**Other differences.**

- Each patient has a `patients` event at their birth date carrying demographics (`gender`, `race`, `deathdate`, ...).
- There is no separate visit level: clinical events carry an `encounter` UUID; group by it to rebuild visits.
- `observations/value` is a string column holding both numbers and text; `observations/type` says which.
- `conditions` timestamps are dates (midnight); most other tables have full date-times.

On a patient, attributes are accessed without the table prefix:

```python
for event in patient.get_events(event_type="conditions"):
    print(event.timestamp, event.code, event.description)
```